# Stage 2: Partitioned Multiple Alignment, Phylogenomics & Empirical Bayesian ASR

## 1. Scientific Overview & Computational Rationale
Constructing an ancestral chloroplast genome requires modeling evolutionary substitutions and indels separately:
- **Partition-Aware Alignment:** The Inverted Repeats evolve significantly slower than Single Copy regions due to copy-dependent gene conversion repair. To eliminate double-counting and rate bias, we align only **one copy of the Inverted Repeat ($\text{IR}_\text{B}$)** alongside the LSC and SSC.
- **Phylogenetic Tree Search:** We infer the species tree using **IQ-TREE 2** under best-fit nucleotide substitution models (`-m GTR+F+I+G4`) with 1,000 Ultrafast Bootstraps, rooting the topology with *Ensete* and *Musella*.
- **Empirical Bayesian Ancestral State Reconstruction (ASR):** Using the `-asr` flag, IQ-TREE 2 calculates site-by-site posterior probability distributions for all internal nodes. We isolate the **Musa crown node (MRCA)**.
- **Codon-Guarded Indel Resolution:** Parsimonious indel resolution guarantees that coding sequences preserve reading frame triplets ($\Delta\text{len} \pmod 3 == 0$), preventing spurious frameshifts.

In [ ]:
# Cell 1: Environment Detection & Workspace Setup
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    %cd /content/musa-hypothetical-structure
else:
    root_path = Path.cwd().resolve()
    if str(root_path) not in sys.path:
        sys.path.insert(0, str(root_path))

import scripts.config as config
from scripts.tool_resolver import resolve_iqtree, resolve_mafft

print("IQ-TREE 2 Path:", resolve_iqtree())
print("MAFFT Path:    ", resolve_mafft())

## 2. Partitioned Alignment & Bayesian ASR via IQ-TREE 2
We run progressive multiple sequence alignment with MAFFT across the normalized partitions (`tier1_LSC.fasta`, `tier1_SSC.fasta`, `tier1_IR_B.fasta`).

IQ-TREE 2 then executes ML tree search and empirical Bayesian ancestral reconstruction, generating `.treefile` and `.state` posterior probability tables.

In [ ]:
# Cell 2: Execute Partitioned Alignment & Ancestral State Reconstruction
from scripts.phylogeny_asr import reconstruct_ancestral_plastome

fasta_path, manifest_path = reconstruct_ancestral_plastome()
print("Ancestral Plastome FASTA:", fasta_path)
print("Ancestral Manifest:      ", manifest_path)

## 3. Inspecting the Inferred Ancestral Plastome Architecture
We inspect the length and composition of the quadripartite partitions of the reconstructed *Musa* MRCA.

In [ ]:
# Cell 3: Summary Architecture of Ancestral Genome
import json
from Bio import SeqIO

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

record = SeqIO.read(fasta_path, "fasta")
seq_str = str(record.seq).upper()
gc_percent = ((seq_str.count("G") + seq_str.count("C")) / len(seq_str)) * 100.0

print(f"Reconstructed Genome ID: {record.id}")
print(f"Total Genome Length:     {len(seq_str):,} bp")
print(f"Plastome GC Content:     {gc_percent:.2f}%")
print("Compartment Coordinates:")
for region, coords in manifest.items():
    if isinstance(coords, dict):
        print(f"  {region.upper():<6}: {coords['start']:>6} .. {coords['end']:>6} ({coords['length']:>6} bp)")